# 01 · EDA · «¿Hoja sana o enferma, y cuándo no fiarnos del modelo? Papas Canarias» (Sprint 14)

**Base de datos:** `PlantVillage-Dataset` (`https://github.com/spMohanty/PlantVillage-Dataset`), con enfoque en el monitoreo sanitario de hojas de papa (*sana* vs. *enferma*) y en identificar **cuándo no fiarnos del modelo** (imágenes borrosas, de bajo contraste/iluminación deficiente o casos ambiguos/fuera de dominio respecto a parcelas de papas canarias).

**Cómo se ejecuta en Google Colab:** *Entorno de ejecución → Reiniciar y ejecutar todo*. `SEMILLA = 42`. Si `PlantVillage-Dataset` no está aún clonado o montado, el cuaderno incluye la celda lista para clonarlo y un respaldo sintético reproducible con `SEMILLA = 42` para que toda la cadena de celdas corra sin errores.

## 0. Preparación y carga de PlantVillage-Dataset

In [ ]:
# Descomenta la siguiente línea en Colab si quieres clonar PlantVillage-Dataset directamente:
# !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git datos/PlantVillage-Dataset

import os, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
np.random.seed(SEMILLA)

plt.rcParams.update({
    "figure.figsize": (9, 3.5),
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False
})

os.makedirs("datos", exist_ok=True)

def md5_archivo(ruta):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8192), b""):
            h.update(bloque)
    return h.hexdigest()

def buscar_carpeta_plantvillage():
    candidatos = [
        "datos/PlantVillage-Dataset",
        "PlantVillage-Dataset",
        "data/PlantVillage-Dataset",
        "../datos/PlantVillage-Dataset"
    ]
    for base in candidatos:
        if os.path.isdir(base):
            for raiz, _, _ in os.walk(base):
                nombre = os.path.basename(raiz).lower()
                if "potato" in nombre or "papa" in nombre:
                    return base, raiz, True
            return base, base, False
    return None, None, False

def cargar_inventario_plantvillage():
    base_dir, foco_dir, hay_papa = buscar_carpeta_plantvillage()
    extensiones = (".jpg", ".jpeg", ".png")
    registros = []

    if base_dir is not None:
        # Si existen subcarpetas específicas de patata/papa, priorizamos esas; si no, tomamos todas las clases de hojas
        subcarpetas = []
        for raiz, _, archivos in os.walk(foco_dir if hay_papa else base_dir):
            imgs = [a for a in archivos if a.lower().endswith(extensiones)]
            if imgs:
                subcarpetas.append((raiz, imgs))

        for raiz, imgs in subcarpetas:
            clase_original = os.path.basename(raiz)
            es_sana = 0 if "healthy" in clase_original.lower() or "sana" in clase_original.lower() else 1
            for nombre_img in imgs[:250]:  # tope ágil por clase para EDA rápido en Colab
                ruta = os.path.join(raiz, nombre_img)
                try:
                    with Image.open(ruta) as im:
                        im_rgb = im.convert("RGB")
                        ancho, alto = im_rgb.size
                        arr = np.asarray(im_rgb, dtype=np.float32) / 255.0
                    brillo = float(arr.mean())
                    contraste = float(arr.std())
                    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
                    verdor = float(((g - r) / (g + r + 1e-5)).mean())
                    registros.append({
                        "archivo": nombre_img,
                        "subcarpeta": clase_original,
                        "es_papa_pv": int("potato" in clase_original.lower() or "papa" in clase_original.lower()),
                        "ancho": ancho,
                        "alto": alto,
                        "brillo": round(brillo, 4),
                        "contraste": round(contraste, 4),
                        "verdor": round(verdor, 4),
                        "md5": md5_archivo(ruta),
                        "hoja_enferma": es_sana
                    })
                except Exception:
                    continue

    if registros:
        df = pd.DataFrame(registros)
        origen = "PlantVillage-Dataset (real)"
        return df, origen

    # Respaldo sintético reproducible con SEMILLA=42 inspirado en clases de hoja sana/enferma
    # y condiciones de niebla/bajo contraste/sombra donde el modelo no es fiable
    n = 1200
    clases_pv = rng.choice(
        ["Potato___healthy", "Potato___Early_blight", "Potato___Late_blight"],
        size=n, p=[0.36, 0.34, 0.30]
    )
    es_sana_arr = np.where(clases_pv == "Potato___healthy", 0, 1)
    brillo = np.where(es_sana_arr == 0, rng.normal(0.54, 0.08, n), rng.normal(0.46, 0.10, n)).clip(0.08, 0.95)
    contraste = np.where(es_sana_arr == 0, rng.normal(0.24, 0.05, n), rng.normal(0.27, 0.06, n)).clip(0.03, 0.45)
    verdor = np.where(es_sana_arr == 0, rng.normal(0.18, 0.06, n), rng.normal(0.03, 0.08, n)).clip(-0.25, 0.45)

    # Introducimos ~8% de casos difíciles / bajo contraste o iluminación anómala (ej. niebla o sombra)
    idx_dificiles = rng.choice(n, size=int(n * 0.08), replace=False)
    contraste[idx_dificiles] = rng.uniform(0.03, 0.09, size=len(idx_dificiles))
    brillo[idx_dificiles[:len(idx_dificiles)//2]] = rng.uniform(0.09, 0.18, size=len(idx_dificiles)//2)

    df = pd.DataFrame({
        "archivo": [f"pv_potato_{i:04d}.jpg" for i in range(n)],
        "subcarpeta": clases_pv,
        "es_papa_pv": np.ones(n, dtype=int),
        "ancho": np.full(n, 256),
        "alto": np.full(n, 256),
        "brillo": np.round(brillo, 4),
        "contraste": np.round(contraste, 4),
        "verdor": np.round(verdor, 4),
        "md5": [hashlib.md5(f"pv_potato_{i}_{clases_pv[i]}_42".encode()).hexdigest() for i in range(n)],
        "hoja_enferma": es_sana_arr
    })
    # Añadimos 10 duplicados MD5 intencionados para comprobar detección de duplicados
    idx_dup = rng.choice(n, size=10, replace=False)
    df.loc[idx_dup, "md5"] = df.loc[idx_dup, "md5"].iloc[0]

    return df, "PlantVillage-Dataset (respaldo sintético SEMILLA=42)"

df_pv, origen_pv = cargar_inventario_plantvillage()
print("Origen:", origen_pv, "· Filas:", len(df_pv))

## 1. Describir los datos de PlantVillage-Dataset

In [ ]:
print("Filas y columnas:", df_pv.shape)
print("Distribución objetivo hoja_enferma (0=sana, 1=enferma):", df_pv["hoja_enferma"].value_counts().to_dict(),
      f"(tasa enferma: {df_pv['hoja_enferma'].mean():.1%})")
print("\nReparto por subcarpeta de PlantVillage-Dataset:")
print(df_pv["subcarpeta"].value_counts().to_string())
df_pv.head(3)

## 2. Calidad, duplicados (MD5), casos de baja fiabilidad y fugas

In [ ]:
n_nulos = int(df_pv.isna().sum().sum())
n_duplicados_fila = int(df_pv.duplicated().sum())
n_duplicados_md5 = int(df_pv["md5"].duplicated().sum())
bajo_contraste = df_pv["contraste"] < 0.10
iluminacion_extrema = (df_pv["brillo"] < 0.20) | (df_pv["brillo"] > 0.85)

print(f"Nulos: {n_nulos} · Duplicados fila: {n_duplicados_fila} · Duplicados MD5: {n_duplicados_md5}")
print(f"Imágenes con bajo contraste (<0.10): {int(bajo_contraste.sum())} ({bajo_contraste.mean():.1%})")
print(f"Imágenes con brillo extremo (<0.20 o >0.85): {int(iluminacion_extrema.sum())} ({iluminacion_extrema.mean():.1%})")

print("\nEstadísticos de características visuales:")
print(df_pv[["ancho", "alto", "brillo", "contraste", "verdor"]].describe().round(3).T.to_string())

print("\nAuditoría de fugas y límites frente a papas canarias:")
print(" - No usar nombres de carpeta/archivo codificados con la etiqueta como variable de entrada.")
print(" - PlantVillage está compuesto principalmente por hojas aisladas sobre fondo uniforme: en campo en Canarias "
      "(niebla, viento, tierra volcánica, variedades locales) los casos de bajo contraste o brillo anómalo requieren revisión humana.")

## 3. Cuatro gráficos clave

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))

df_pv["subcarpeta"].value_counts().plot.bar(ax=axes[0, 0], color="#7c98ab", rot=20)
axes[0, 0].set_title("1. Muestras por subcarpeta (PlantVillage)")
axes[0, 0].set_ylabel("Cantidad")

df_pv.boxplot(column="verdor", by="hoja_enferma", ax=axes[0, 1])
axes[0, 1].set_title("2. Índice de verdor (0=sana, 1=enferma)")
axes[0, 1].set_xlabel("Estado")
axes[0, 1].set_ylabel("Verdor")
plt.suptitle("")

axes[1, 0].scatter(df_pv["contraste"], df_pv["brillo"], c=df_pv["hoja_enferma"], cmap="coolwarm", alpha=0.4, s=14)
axes[1, 0].axvline(0.10, color="#c85a32", linestyle="--", linewidth=1.2, label="Umbral contraste=0.10")
axes[1, 0].set_title("3. Contraste vs. brillo (rojo=enferma)")
axes[1, 0].set_xlabel("Contraste")
axes[1, 0].set_ylabel("Brillo")
axes[1, 0].legend(fontsize=8)

tasa_sub = df_pv.groupby("subcarpeta")["hoja_enferma"].mean()
tasa_sub.plot.barh(ax=axes[1, 1], color="#4a6b82")
axes[1, 1].set_title("4. Proporción enferma por subcarpeta")
axes[1, 1].set_xlabel("Proporción")

plt.tight_layout()
plt.show()

## 4. Hallazgo que cambia el enfoque e informe de calidad / linaje

In [ ]:
import json, sys, sklearn

hallazgo = (
    "Las hojas enfermas presentan menor índice de verdor medio, pero las imágenes con bajo contraste (<0.10) "
    "o iluminación extrema concentran mayor ambigüedad visual y riesgo de error al transferir PlantVillage a campo. "
    "Cambia el enfoque: el modelo no debe decidir en solitario en fotos de baja calidad visual o duda alta; "
    "debe marcar esos casos para revisión humana."
)

informe = {
    "proyecto": "¿Hoja sana o enferma, y cuándo no fiarnos del modelo? Papas Canarias",
    "base_datos": "https://github.com/spMohanty/PlantVillage-Dataset",
    "origen_ejecucion": origen_pv,
    "n_filas": int(len(df_pv)),
    "n_duplicados_md5": n_duplicados_md5,
    "n_bajo_contraste": int(bajo_contraste.sum()),
    "hallazgo_cambia_enfoque": hallazgo,
    "linaje": [
        "Fuente: PlantVillage-Dataset (spMohanty) o respaldo sintético SEMILLA=42 si no está descargado",
        "EDA: cálculo de brillo, contraste, verdor, hash MD5, detección de duplicados y casos de baja fiabilidad"
    ],
    "versiones": f"Python {sys.version.split()[0]}, numpy {np.__version__}, pandas {pd.__version__}, scikit-learn {sklearn.__version__}"
}
print(hallazgo)
print("\n" + json.dumps(informe, ensure_ascii=False, indent=2))